In [35]:
import os
from getpass import getpass

from dotenv import load_dotenv



In [36]:

load_dotenv()

def set_api_key_if_not_present(key_name, prompt_message=""):
    if len(prompt_message) == 0:
        prompt_message=key_name
    if key_name not in os.environ or not os.environ[key_name]:
        os.environ[key_name] = getpass.getpass(prompt_message)

set_api_key_if_not_present("OPENAI_API_KEY")

# Data Preparation

First, we will read in the transcripts of the videos and convert them to Documents
with appropriate metadata.

In [37]:
filename = "../test.json"


In [38]:
import json

data = json.load(open(filename, "rb"))
data[0]

{'video_id': 4157,
 'title': 'Get organized with layer groups',
 'desc': 'Learn some great tips for working with layers.',
 'length': '00:04:05.78',
 'url': 'https://videos-tv.adobe.com/2013-07-23/f65b5a0ef188ba5e5a96df93a8ead3cf.mp4',
 'transcripts': [{'sent_id': 0,
   'sent': "At any time when you're working in Adobe Photoshop on a complicated subject the possibility exists of chaos ensuing.",
   'begin': 1.27,
   'end': 9.97},
  {'sent_id': 1,
   'sent': "Well, you know what chaos there is, don't you?",
   'begin': 10.16,
   'end': 12.01},
  {'sent_id': 2,
   'sent': "It's my studio in the middle of a project.",
   'begin': 12.009999,
   'end': 14.23},
  {'sent_id': 3,
   'sent': 'What I would like to do is reduce the clutter, reduce the chaos.',
   'begin': 14.58,
   'end': 17.719999},
  {'sent_id': 4,
   'sent': 'Let me move Layers over here again, I would say that is not really necessary, but it makes life easier.',
   'begin': 17.719999,
   'end': 23.15},
  {'sent_id': 5,
   'se

In [39]:
from langchain_core.document_loaders import BaseLoader
from typing import List, Dict, Iterator
from langchain_core.documents import Document

class VideoTranscriptBulkLoader(BaseLoader):
    """Loads video transcripts as a bulk into documents"""

    def __init__(self, json_payload:List[Dict]):

        self.json_payload = json_payload
        
    def lazy_load(self) -> Iterator[Document]:
        """Lazy loader that returns an iterator"""
        
        for video in self.json_payload:
            metadata = dict(video)
            metadata.pop("transcripts", None)
            metadata.pop("qa", None)
            # Rename 'url' key to 'source' in metadata if it exists
            if "url" in metadata:
                metadata["source"] = metadata.pop("url")
            yield Document(
                page_content = "\n".join(t["sent"] for t in video["transcripts"]),
                metadata = metadata
            )

class VideoTranscriptLoader(BaseLoader):
    """Loads video transcripts as individual chunks into documents"""

    def __init__(self, json_payload:List[Dict]):

        self.json_payload = json_payload
        
    def lazy_load(self) -> Iterator[Document]:
        """Lazy loader that returns an iterator"""
        
        for video in self.json_payload:
            metadata = dict(video)
            transcripts = metadata.pop("transcripts", None)
            metadata.pop("qa", None)
            # Rename 'url' key to 'source' in metadata if it exists
            if "url" in metadata:
                metadata["source"] = metadata.pop("url")
            for transcript in transcripts:
                yield Document(
                    page_content = transcript["sent"],
                    metadata = metadata | {
                        "time_start":transcript["begin"],
                        "time_end":transcript["end"],
                                           }
                )


docs_full_transcript = VideoTranscriptBulkLoader(data).load()
docs_chunks_verbatim = VideoTranscriptLoader(data).load()


In [40]:
from pprint import pp

pp(docs_full_transcript[0])
pp(docs_chunks_verbatim[0])

Document(metadata={'video_id': 4157, 'title': 'Get organized with layer groups', 'desc': 'Learn some great tips for working with layers.', 'length': '00:04:05.78', 'source': 'https://videos-tv.adobe.com/2013-07-23/f65b5a0ef188ba5e5a96df93a8ead3cf.mp4'}, page_content='At any time when you\'re working in Adobe Photoshop on a complicated subject the possibility exists of chaos ensuing.\nWell, you know what chaos there is, don\'t you?\nIt\'s my studio in the middle of a project.\nWhat I would like to do is reduce the clutter, reduce the chaos.\nLet me move Layers over here again, I would say that is not really necessary, but it makes life easier.\nI have an Andy\'s Funny Face character.\nEach one of the pieces of the face are in separate layers.\nNothing wrong with that.\nIt doesn\'t really hurt a thing, gives me total control.\nBut I would like to reduce the clutter in my Layers panel by creating something called a Group.\nSo let\'s do this a couple of ways.\nNumber one, we can come down 

## R - retrieval

Let's hit it with a semantic chunker.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [42]:
text_splitter = SemanticChunker(embeddings)

docs_chunks_semantic = text_splitter.split_documents(docs_full_transcript)

pp(docs_chunks_semantic[0])

print(len(docs_chunks_semantic))
print(len(docs_chunks_verbatim))

Document(metadata={'video_id': 4157, 'title': 'Get organized with layer groups', 'desc': 'Learn some great tips for working with layers.', 'length': '00:04:05.78', 'source': 'https://videos-tv.adobe.com/2013-07-23/f65b5a0ef188ba5e5a96df93a8ead3cf.mp4'}, page_content="At any time when you're working in Adobe Photoshop on a complicated subject the possibility exists of chaos ensuing. Well, you know what chaos there is, don't you? It's my studio in the middle of a project. What I would like to do is reduce the clutter, reduce the chaos. Let me move Layers over here again, I would say that is not really necessary, but it makes life easier. I have an Andy's Funny Face character. Each one of the pieces of the face are in separate layers. Nothing wrong with that.")
40
485


In [43]:
for semantic_chunk in docs_chunks_semantic:
    same_video_chunks = list( filter(
                               lambda verbatim_chunk: (verbatim_chunk.metadata['video_id'] == semantic_chunk.metadata['video_id']) &
                                    (verbatim_chunk.page_content in semantic_chunk.page_content),
                                    docs_chunks_verbatim
                               ) )
    semantic_chunk.metadata["speech_start_stop_times"] = [ (t.metadata["time_start"],t.metadata["time_end"]) for t in same_video_chunks ]
    semantic_chunk.metadata["start"],semantic_chunk.metadata["stop"] = ( semantic_chunk.metadata["speech_start_stop_times"][0][0],semantic_chunk.metadata["speech_start_stop_times"][-1][-1] )


In [44]:
pp(docs_chunks_semantic[0])

Document(metadata={'video_id': 4157, 'title': 'Get organized with layer groups', 'desc': 'Learn some great tips for working with layers.', 'length': '00:04:05.78', 'source': 'https://videos-tv.adobe.com/2013-07-23/f65b5a0ef188ba5e5a96df93a8ead3cf.mp4', 'speech_start_stop_times': [(1.27, 9.97), (10.16, 12.01), (12.009999, 14.23), (14.58, 17.719999), (17.719999, 23.15), (23.8, 27.27), (27.41, 32.349999), (32.9, 33.749999)], 'start': 1.27, 'stop': 33.749999}, page_content="At any time when you're working in Adobe Photoshop on a complicated subject the possibility exists of chaos ensuing. Well, you know what chaos there is, don't you? It's my studio in the middle of a project. What I would like to do is reduce the clutter, reduce the chaos. Let me move Layers over here again, I would say that is not really necessary, but it makes life easier. I have an Andy's Funny Face character. Each one of the pieces of the face are in separate layers. Nothing wrong with that.")


In [45]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

collection_name = f"{filename}_qdrant"

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
)

In [46]:
_ = vector_store.add_documents(documents=docs_chunks_semantic)

In [47]:
retriever = vector_store.as_retriever(search_kwargs={"k":2})

def retrieve(state):
    retrieved_docs = retriever.invoke(state["question"])
    return {"context":retrieved_docs}


In [48]:
a = retrieve({"question":"What is a layer?"})
[ pp(d.page_content) for d in a["context"] ]

("Layers are the building blocks of any image in Photoshop CC. So, it's "
 "important to understand, what layers are and why to use them - which we'll "
 "cover in this video. If you're following along, open this layered image from "
 'the downloadable practice files for this tutorial. You might think of layers '
 'like separate flat pints of glass, stacked one on top of the other. Each '
 'layer contains separate pieces of content. To get a sense of how layers are '
 "constructed, let's take a look at this Layers panel. I've closed my other "
 'panels, so that we can focus on the Layers panel. But you can skip that. By '
 "the way: If your Layers panel isn't showing, go up to the Window menu and "
 'choose Layers from there. The Layers panel is where you go to select and '
 'work with layers. In this image there are 4 layers, each with separate '
 'content. If you click the Eye icon to the left of a layer, you can toggle '
 "the visibility of that layer off and on. So, I'm going to tu

[None, None]

## A - Augmentation

We need to populate a prompt for LLM.


In [49]:
from langchain.prompts import ChatPromptTemplate

SYSTEM_PROMPT = """\
You are a helpful an expert on Photoshop and your goal is to help users
gain knowledge from a database of training videos. 
You answer questions based on provided context. 
Your answers use emojis for emphasis.

IMPORTANT: You must only use the provided context, and cannot use your own knowledge.
If there is no context that corresponds to the query, respond by saying
"I don't know. This is not available in our training library."

Most of the users questions will be in the form:
"How can I do ..."
or
"What is ..."

When appropriate, provide your answers in a step-by-step form.
ALWAYS list the URL and the title of the reference video.
NEVER invent the explanation. ALWAYS use ONLY the context information.

"""

RAG_PROMPT="""\

### Question
{question}

NEVER invent the explanation. ALWAYS use ONLY the context information.

### Context
{context}


"""

rag_prompt = ChatPromptTemplate(
    [("system",SYSTEM_PROMPT), 
     ("human",RAG_PROMPT)
     ]
    )

## Generation

We will use a 4.1-nano to generate answers.

In [50]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano",temperature=0)

In [51]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])

  references = [ 
                {k: doc.metadata[k] for k in ("title","source","start","stop")} 
                for doc in state["context"] 
  ] 


  messages = rag_prompt.format_messages(question=state["question"], 
                                        context=docs_content)
  response = llm.invoke(messages)
  retval = {"response":f"{response.content}\n\n**References**:\n{json.dumps(references,indent=2)}",
            "context":state["context"]}
  
  return retval


In [52]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict,Annotated
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_openai.chat_models import ChatOpenAI
import operator

class State(TypedDict):
    question: str
    context: List[Document]
    response: str
        
graph_builder = StateGraph(State).add_sequence([retrieve, generate ])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [53]:
from langchain.schema.output_parser import StrOutputParser
response = graph.invoke({"question" : "What is the layer in Photoshop"})

In [54]:
response.keys()

dict_keys(['question', 'context', 'response'])

In [55]:
type(response)

langgraph.pregel.io.AddableValuesDict

In [56]:
pp(response)

{'question': 'What is the layer in Photoshop',
 'context': [Document(metadata={'video_id': 19172, 'title': 'Understand layers', 'desc': 'Learn what layers are and why they are so useful.', 'length': '00:04:44.75', 'source': 'https://images-tv.adobe.com/avp/vr/b758b4c4-2a74-41f4-8e67-e2f2eab83c6a/f810fc5b-2b04-4e23-8fa4-5c532e7de6f8/e268fe4d-e5c7-415c-9f5c-d34d024b14d8_20170727011753.1280x720at2400_h264.mp4', 'speech_start_stop_times': [[0.47, 3.41], [3.81, 9.13], [9.309999, 15.01], [15.299999, 20.57], [20.88, 23.3], [23.83, 27.93], [29.38, 32.79], [32.96, 33.92], [34.43, 40.21], [41.91, 45.37], [45.88, 49.01], [49.54, 55.130001], [55.72, 58.49], [58.72, 62.14]], 'start': 0.47, 'stop': 62.14, '_id': 'faaf3453d9494e15be8b72d65b91a44d', '_collection_name': '../test.json_qdrant'}, page_content="Layers are the building blocks of any image in Photoshop CC. So, it's important to understand, what layers are and why to use them - which we'll cover in this video. If you're following along, open 

In [57]:
response.keys()

dict_keys(['question', 'context', 'response'])